# Markov Chains: The Memoryless State Machine

A Markov chain models a system that moves between states using probabilities.

Andrey Markov developed the idea in the early 1900s while studying patterns in text. Markov chains now model weather, queues, board games, genetics, finance, web behavior, and any system where the next step can be predicted from the current state.

The key rule is simple: **the next state depends only on the current state**, not the full history.

In this notebook, we will build a small object-oriented Markov chain that simulates a student's day moving between study, snacks, walks, and rest.

<details>
<summary>Big idea</summary>

A Markov chain is a probability machine. Each state has outgoing probabilities that say where the system might go next.

</details>

## 1. The Mental Model

A Markov chain needs:

- **states**: places the system can be
- **transition probabilities**: chances of moving from one state to another
- **current state**: where the system is now
- **step rule**: randomly choose the next state from the current state's probabilities

If every row of transition probabilities sums to `1`, the chain is ready to simulate.

<details>
<summary>Memoryless hint</summary>

If the current state is `Study`, the chain only looks at the `Study` row. It does not care how you got there.

</details>

## 2. Build the Objects

Implementation plan:

1. `State` names one possible condition.
2. `Transition` stores a weighted move from one state to another.
3. `WalkStep` records one simulated move.
4. `DistributionStep` records how probabilities spread over time.
5. `MarkovChain` validates rows, samples next states, and updates distributions.
6. `WalkReplay` and `DistributionReplay` print the story.

<details>
<summary>Implementation hint</summary>

Sampling uses a random number between `0` and `1`. We walk across cumulative probabilities until the random pick lands inside one transition's range.

</details>

**Object model.** Define `State`, `Transition`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass

import random

@dataclass(frozen=True)
class State:
    name: str

    def __str__(self) -> str:
        return self.name

@dataclass(frozen=True)
class Transition:
    source: State
    target: State
    probability: float

    def __str__(self) -> str:
        return f"{self.source} -> {self.target} ({self.probability:.0%})"


**Trace model.** Define `WalkStep`, `DistributionStep`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass(frozen=True)
class WalkStep:
    step_number: int
    current: State
    next_state: State
    roll: float
    options: tuple[Transition, ...]

@dataclass(frozen=True)
class DistributionStep:
    step_number: int
    distribution: dict[State, float]


**Object model.** Define `MarkovChain`, the named objects used by the next examples.


In [ ]:
class MarkovChain:
    def __init__(self, states: list[State], transitions: list[Transition], seed: int = 3):
        self.states = states
        self.rng = random.Random(seed)
        self.transitions_by_state: dict[State, list[Transition]] = {state: [] for state in states}

        for transition in transitions:
            self.transitions_by_state[transition.source].append(transition)

        self._validate()

    def _validate(self) -> None:
        for state in self.states:
            options = self.transitions_by_state[state]
            if not options:
                raise ValueError(f"State {state} has no outgoing transitions.")

            total = sum(transition.probability for transition in options)
            if abs(total - 1.0) > 0.000001:
                raise ValueError(f"Probabilities leaving {state} sum to {total:.3f}, not 1.")

    def next_state(self, current: State) -> tuple[State, float, tuple[Transition, ...]]:
        roll = self.rng.random()
        running_total = 0.0
        options = tuple(self.transitions_by_state[current])

        for transition in options:
            running_total += transition.probability
            if roll <= running_total:
                return transition.target, roll, options

        return options[-1].target, roll, options

    def walk(self, start: State, steps: int) -> list[WalkStep]:
        current = start
        story: list[WalkStep] = []

        for step_number in range(1, steps + 1):
            next_state, roll, options = self.next_state(current)
            story.append(WalkStep(step_number, current, next_state, roll, options))
            current = next_state

        return story

    def next_distribution(self, distribution: dict[State, float]) -> dict[State, float]:
        updated = {state: 0.0 for state in self.states}

        for state, amount in distribution.items():
            for transition in self.transitions_by_state[state]:
                updated[transition.target] += amount * transition.probability

        return updated

    def evolve_distribution(self, start: State, steps: int) -> list[DistributionStep]:
        distribution = {state: 0.0 for state in self.states}
        distribution[start] = 1.0
        story = [DistributionStep(0, distribution.copy())]

        for step_number in range(1, steps + 1):
            distribution = self.next_distribution(distribution)
            story.append(DistributionStep(step_number, distribution.copy()))

        return story

    def transition_table(self) -> str:
        lines = []
        for state in self.states:
            pieces = [f"{transition.target}:{transition.probability:.0%}" for transition in self.transitions_by_state[state]]
            lines.append(f"{state}: " + ", ".join(pieces))
        return "\n".join(lines)


## 3. Build a Student-Day Chain

Our states describe what a student is doing next:

- `Study`
- `Snack`
- `Walk`
- `Rest`

Each state has its own transition probabilities.

<details>
<summary>Why rows must sum to 1</summary>

When the chain is in a state, one of that state's outgoing transitions must happen. The probabilities are a full menu of possible next moves.

</details>

**Example state.** Create `study`, `snack`, `walk`, `rest`, and related helpers, the concrete values used in the next run.


In [ ]:
study = State("Study")

snack = State("Snack")

walk = State("Walk")

rest = State("Rest")

states = [study, snack, walk, rest]


**Example state.** Create `transitions`, `student_chain`, the concrete values used in the next run.


In [ ]:
transitions = [
    Transition(study, study, 0.45),
    Transition(study, snack, 0.25),
    Transition(study, walk, 0.20),
    Transition(study, rest, 0.10),
    Transition(snack, study, 0.30),
    Transition(snack, snack, 0.10),
    Transition(snack, walk, 0.40),
    Transition(snack, rest, 0.20),
    Transition(walk, study, 0.35),
    Transition(walk, snack, 0.15),
    Transition(walk, walk, 0.20),
    Transition(walk, rest, 0.30),
    Transition(rest, study, 0.50),
    Transition(rest, snack, 0.20),
    Transition(rest, walk, 0.10),
    Transition(rest, rest, 0.20),
]

student_chain = MarkovChain(states, transitions, seed=11)

print("Transition table:")

print(student_chain.transition_table())


## 4. Simulate One Random Walk

A random walk follows one possible path through the chain. Because the choices are random, this is just one story the model can generate.

<details>
<summary>Randomness hint</summary>

The notebook uses a fixed random seed, so your output stays stable while you learn the mechanics.

</details>

In [3]:
class WalkReplay:
    def __init__(self, steps: list[WalkStep]):
        self.steps = steps

    def show(self) -> None:
        route = [self.steps[0].current.name] if self.steps else []

        for step in self.steps:
            route.append(step.next_state.name)
            option_text = ", ".join(f"{transition.target}:{transition.probability:.0%}" for transition in step.options)
            print(f"Step {step.step_number:>2}: {step.current} -> {step.next_state} | roll={step.roll:.3f}")
            print(f"          options from {step.current}: {option_text}")

        print("\nRoute:")
        print(" -> ".join(route))


walk_steps = student_chain.walk(start=study, steps=12)
WalkReplay(walk_steps).show()

Step  1: Study -> Snack | roll=0.452
          options from Study: Study:45%, Snack:25%, Walk:20%, Rest:10%
Step  2: Snack -> Walk | roll=0.560
          options from Snack: Study:30%, Snack:10%, Walk:40%, Rest:20%
Step  3: Walk -> Rest | roll=0.924
          options from Walk: Study:35%, Snack:15%, Walk:20%, Rest:30%
Step  4: Rest -> Study | roll=0.466
          options from Rest: Study:50%, Snack:20%, Walk:10%, Rest:20%
Step  5: Study -> Snack | roll=0.508
          options from Study: Study:45%, Snack:25%, Walk:20%, Rest:10%
Step  6: Snack -> Walk | roll=0.587
          options from Snack: Study:30%, Snack:10%, Walk:40%, Rest:20%
Step  7: Walk -> Study | roll=0.185
          options from Walk: Study:35%, Snack:15%, Walk:20%, Rest:30%
Step  8: Study -> Snack | roll=0.512
          options from Study: Study:45%, Snack:25%, Walk:20%, Rest:10%
Step  9: Snack -> Walk | roll=0.630
          options from Snack: Study:30%, Snack:10%, Walk:40%, Rest:20%
Step 10: Walk -> Rest | roll=0.793
   

## 5. Evolve a Probability Distribution

A random walk shows one possible future. A distribution shows all possible futures at once.

Starting from `100% Study`, we repeatedly apply the transition table and watch probability mass spread across states.

<details>
<summary>Stationary distribution</summary>

Many Markov chains settle into a long-run balance called a stationary distribution. After enough steps, the distribution changes less and less.

</details>

In [4]:
class DistributionReplay:
    def __init__(self, steps: list[DistributionStep]):
        self.steps = steps

    def show(self, selected_steps: list[int]) -> None:
        for step_number in selected_steps:
            step = self.steps[step_number]
            pieces = [f"{state.name}:{probability:>6.1%}" for state, probability in step.distribution.items()]
            print(f"After {step.step_number:>2} step(s): " + " | ".join(pieces))


distribution_steps = student_chain.evolve_distribution(start=study, steps=20)
DistributionReplay(distribution_steps).show([0, 1, 2, 5, 10, 20])

After  0 step(s): Study:100.0% | Snack:  0.0% | Walk:  0.0% | Rest:  0.0%
After  1 step(s): Study: 45.0% | Snack: 25.0% | Walk: 20.0% | Rest: 10.0%
After  2 step(s): Study: 39.8% | Snack: 18.8% | Walk: 24.0% | Rest: 17.5%
After  5 step(s): Study: 40.9% | Snack: 19.0% | Walk: 22.0% | Rest: 18.1%
After 10 step(s): Study: 40.9% | Snack: 19.0% | Walk: 22.0% | Rest: 18.1%
After 20 step(s): Study: 40.9% | Snack: 19.0% | Walk: 22.0% | Rest: 18.1%


## 6. Experiments

Change the transition probabilities and watch the long-run balance move.

<details>
<summary>Experiment hint</summary>

Markov chains are sensitive to repeated probabilities. A small preference in one row can compound over many steps.

</details>

In [5]:
def long_run_distribution(chain: MarkovChain, start: State, steps: int = 40) -> dict[State, float]:
    return chain.evolve_distribution(start=start, steps=steps)[-1].distribution


def summarize_distribution(label: str, distribution: dict[State, float]) -> None:
    pieces = [f"{state.name}:{probability:>6.1%}" for state, probability in distribution.items()]
    print(f"{label:<18} " + " | ".join(pieces))


study_heavy_transitions = [
    Transition(study, study, 0.65), Transition(study, snack, 0.15), Transition(study, walk, 0.15), Transition(study, rest, 0.05),
    Transition(snack, study, 0.45), Transition(snack, snack, 0.10), Transition(snack, walk, 0.30), Transition(snack, rest, 0.15),
    Transition(walk, study, 0.50), Transition(walk, snack, 0.10), Transition(walk, walk, 0.15), Transition(walk, rest, 0.25),
    Transition(rest, study, 0.65), Transition(rest, snack, 0.15), Transition(rest, walk, 0.05), Transition(rest, rest, 0.15),
]

rest_heavy_transitions = [
    Transition(study, study, 0.30), Transition(study, snack, 0.20), Transition(study, walk, 0.15), Transition(study, rest, 0.35),
    Transition(snack, study, 0.20), Transition(snack, snack, 0.10), Transition(snack, walk, 0.25), Transition(snack, rest, 0.45),
    Transition(walk, study, 0.20), Transition(walk, snack, 0.10), Transition(walk, walk, 0.20), Transition(walk, rest, 0.50),
    Transition(rest, study, 0.30), Transition(rest, snack, 0.15), Transition(rest, walk, 0.05), Transition(rest, rest, 0.50),
]

study_heavy_chain = MarkovChain(states, study_heavy_transitions, seed=21)
rest_heavy_chain = MarkovChain(states, rest_heavy_transitions, seed=21)

summarize_distribution("Original", long_run_distribution(student_chain, study))
summarize_distribution("Study-heavy", long_run_distribution(study_heavy_chain, study))
summarize_distribution("Rest-heavy", long_run_distribution(rest_heavy_chain, study))

Original           Study: 40.9% | Snack: 19.0% | Walk: 22.0% | Rest: 18.1%
Study-heavy        Study: 59.9% | Snack: 13.5% | Walk: 16.0% | Rest: 10.6%
Rest-heavy         Study: 27.2% | Snack: 15.0% | Walk: 12.6% | Rest: 45.2%


## 7. Matrix View

A Markov chain is often stored as a transition matrix. Rows are current states, columns are next states, and each row sums to `1`.

<details>
<summary>Matrix hint</summary>

The `Study` row means: if the current state is `Study`, these are the probabilities for the next state.

</details>

In [6]:
def transition_matrix(chain: MarkovChain) -> list[list[float]]:
    index_by_state = {state: index for index, state in enumerate(chain.states)}
    matrix = [[0.0 for _ in chain.states] for _ in chain.states]

    for row_state in chain.states:
        row_index = index_by_state[row_state]
        for transition in chain.transitions_by_state[row_state]:
            column_index = index_by_state[transition.target]
            matrix[row_index][column_index] = transition.probability

    return matrix


matrix = transition_matrix(student_chain)
header = "from\\to".ljust(10) + "".join(state.name.rjust(9) for state in states)
print(header)
for state, row in zip(states, matrix):
    row_text = "".join(f"{value:>8.0%} " for value in row)
    print(state.name.ljust(10) + row_text + f"| row sum={sum(row):.1f}")

from\to       Study    Snack     Walk     Rest
Study          45%      25%      20%      10% | row sum=1.0
Snack          30%      10%      40%      20% | row sum=1.0
Walk           35%      15%      20%      30% | row sum=1.0
Rest           50%      20%      10%      20% | row sum=1.0


## What You Should Remember

Markov chains model probabilistic state changes:

- The next state depends only on the current state.
- Each state has a row of outgoing probabilities.
- A random walk shows one possible future.
- A distribution shows all possible futures at once.
- Repeated updates can settle into a long-run balance.
- Transition matrices are the compact math form of the same idea.

<details>
<summary>Where this shows up</summary>

Markov chains show up in PageRank, language models, queueing systems, weather models, board games, finance, biology, and simulation.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Model a system whose next state depends only on its current state.

**Interactive animation target.** Animate probability mass flowing through a transition graph over time.

**Correctness handle.** The probability vector sums to one after every transition.

**Complexity handle.** One dense transition step is O(n^2); sparse chains can be much cheaper.

**Failure mode to test.** Reducibility or periodicity can prevent simple convergence to one stable distribution.

**Studio task.** Alter one transition probability and compare the long-run state distribution.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
